# Definitions

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from typing import List
import torch.nn.functional as F
import torch.optim as optim
import torchaudio
import pandas as pd
import os

In [2]:
AUDIO_DIR = "../../data_processing/separate_scripts/golden_audio"
LABEL_DIR = "../../data_processing/separate_scripts/golden_breaks"
INDEX_SET = [0, 6, 19]  # 16
interval_width = 20  # ms
NUM_CLASSES = 2

In [3]:
class BreakDataset(Dataset):
    def __init__(self, indices: List[int] = INDEX_SET, audio_dir: str = AUDIO_DIR, label_dir: str = LABEL_DIR):
        self.indices = indices
        self.audio_dir = audio_dir
        self.label_dir = label_dir

        self.data = []
        for idx in indices:
            # Load audio and convert to spectrogram
            audio_path = os.path.join(audio_dir, f"{idx}.mp3")
            waveform, sr = torchaudio.load(audio_path)

            # Ensure mono channel
            waveform = waveform.mean(dim=0, keepdim=True)

            # Compute mel spectrogram
            mel_spec = torchaudio.transforms.MelSpectrogram(
                sample_rate=sr,
                hop_length=int(sr * interval_width / 1000),
                n_mels=32
            )(waveform)

            mel_spec = mel_spec.squeeze(0)

            # Add normalization
            mel_spec = torch.log(mel_spec + 1e-9)  # Log scale
            mel_spec = (mel_spec - mel_spec.mean()) / (mel_spec.std() + 1e-9)  # Normalize

            # Load labels
            label_path = os.path.join(label_dir, f"{idx}.csv")
            df = pd.read_csv(label_path)
            labels = torch.tensor(df['break'].values, dtype=torch.bool)

            # Truncate to last break + 2
            last_break = labels.nonzero(as_tuple=True)[0][-1]
            mel_spec = mel_spec[:, :last_break + 2]
            labels = labels[:last_break + 2]

            print(f"Index: {idx}, Mel shape: {mel_spec.shape}, Labels shape: {labels.shape}")

            self.data.append((mel_spec, labels))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        return self.data[idx]

In [5]:
class CNNRNN(nn.Module):
    def __init__(self, n_mels=32, hidden_size=128, num_layers=3):
        super(CNNRNN, self).__init__()
        # self.conv = nn.Sequential(
        #     nn.Conv1d(n_mels, 64, kernel_size=3, padding=1),
        #     nn.ReLU(),
        #     nn.Conv1d(64, 32, kernel_size=3, padding=1),
        #     nn.ReLU()
        # )
        # self.rnn = nn.LSTM(
        #     input_size=32,
        #     hidden_size=hidden_size,
        #     num_layers=num_layers,
        #     batch_first=True,
        #     bidirectional=True
        # )

        self.conv = nn.Sequential(
            nn.Conv1d(n_mels, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Conv1d(64, 32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU()
        )

        self.rnn = nn.LSTM(
            input_size=32,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.2 if num_layers > 1 else 0
        )
        
        self.fc = nn.Sequential(
            nn.Linear(hidden_size * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, NUM_CLASSES)
        )

    def forward(self, x, mask):
        x = self.conv(x)
        x = x.permute(0, 2, 1)
        x, _ = self.rnn(x)
        x = self.fc(x)
        x = x * mask.unsqueeze(-1)
        return x

# Training

In [4]:
from sklearn.metrics import confusion_matrix

In [6]:
MODEL_PATH = 'best_model.pth'
# Split data into train and validation sets
total_indices = len(INDEX_SET)
train_size = int(0.75 * total_indices)
train_indices = INDEX_SET[:train_size]
val_indices = INDEX_SET[train_size:]
train_dataset = BreakDataset(train_indices, AUDIO_DIR, LABEL_DIR)
val_dataset = BreakDataset(val_indices, AUDIO_DIR, LABEL_DIR)

train_dataloader = DataLoader(train_dataset, batch_size=1, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=1, shuffle=False)

Index: 0, Mel shape: torch.Size([32, 11737]), Labels shape: torch.Size([11737])
Index: 6, Mel shape: torch.Size([32, 4970]), Labels shape: torch.Size([4970])
Index: 19, Mel shape: torch.Size([32, 3642]), Labels shape: torch.Size([3642])


In [7]:
# Initialize the model, optimizer, and loss function
model = CNNRNN()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=2)

# Calculate class weights for the loss function
all_labels = torch.cat([labels for _, labels in train_dataset])
pos_weight = torch.tensor([(all_labels == 0).sum() / (all_labels == 1).sum()])
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Training loop with validation and early stopping
epochs = 30  # Increased epochs since we have early stopping
best_loss = float('inf')
patience = 10
patience_counter = 0

for epoch in range(epochs):
    # Training phase
    model.train()
    train_loss = 0
    train_batches = 0

    for batch in train_dataloader:
        mel_spec, labels = batch
        max_length = mel_spec.shape[2]
        mel_specs = []
        labels_list = []

        for i in range(mel_spec.shape[0]):
            mel_specs.append(F.pad(mel_spec[i], (0, max_length - mel_spec[i].shape[1])))
            labels_list.append(F.pad(labels[i], (0, max_length - labels[i].shape[0])))

        mel_specs = torch.stack(mel_specs)
        labels = torch.stack(labels_list)
        mask = (mel_specs != 0).any(dim=1).float()

        optimizer.zero_grad()
        output = model(mel_specs, mask)
        loss = criterion(output.view(-1, 2)[:, 1], labels.view(-1).float())
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_batches += 1

    avg_train_loss = train_loss / train_batches

    # Validation phase
    model.eval()
    val_loss = 0
    val_batches = 0
    all_predictions = []
    all_true_labels = []

    with torch.no_grad():
        for batch in val_dataloader:
            mel_spec, labels, _ = batch
            max_length = mel_spec.shape[2]
            mel_specs = []
            labels_list = []

            for i in range(mel_spec.shape[0]):
                mel_specs.append(F.pad(mel_spec[i], (0, max_length - mel_spec[i].shape[1])))
                labels_list.append(F.pad(labels[i], (0, max_length - labels[i].shape[0])))

            mel_specs = torch.stack(mel_specs)
            labels = torch.stack(labels_list)
            mask = (mel_specs != 0).any(dim=1).float()

            output = model(mel_specs, mask)
            loss = criterion(output.view(-1, 2)[:, 1], labels.view(-1).float())

            # Get predictions (threshold at 0.5)
            predictions = (torch.sigmoid(output.view(-1, 2)[:, 1]) > 0.5).float()

            # Store predictions and true labels
            all_predictions.extend(predictions.cpu().numpy())
            all_true_labels.extend(labels.view(-1).cpu().numpy())

            val_loss += loss.item()
            val_batches += 1

    avg_val_loss = val_loss / val_batches

    # Calculate confusion matrix
    cm = confusion_matrix(all_true_labels, all_predictions)
    print("\nConfusion Matrix:")
    print("TN FP")
    print("FN TP")
    print(cm)

    # Calculate metrics
    tn, fp, fn, tp = cm.ravel()
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-score: {f1:.4f}")


    # Learning rate scheduling
    scheduler.step(avg_val_loss)

    print(f"Epoch {epoch + 1}")
    print(f"Training Loss: {avg_train_loss:.4f}")
    print(f"Validation Loss: {avg_val_loss:.4f}")
    print(f"Learning Rate: {optimizer.param_groups[0]['lr']:.6f}")

    # Early stopping check
    if avg_val_loss < best_loss:
        best_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), MODEL_PATH)
        print("Saved new best model!")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping triggered!")
            break

    print("-" * 50)

print("Training completed!")


Index: 0, Mel shape: torch.Size([32, 11737]), Labels shape: torch.Size([11737])
Index: 6, Mel shape: torch.Size([32, 4970]), Labels shape: torch.Size([4970])
Index: 19, Mel shape: torch.Size([32, 3642]), Labels shape: torch.Size([3642])


/home/eizigi/.local/share/virtualenvs/CS_541_Project-RTvQqKvc/lib/python3.10/site-packages/torch/autograd/graph.py:825: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 803: system has unsupported display driver / cuda driver combination (Triggered internally at ../c10/cuda/CUDAFunctions.cpp:108.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


ValueError: not enough values to unpack (expected 3, got 2)

# Inference

In [7]:
import torch
import torchaudio
import pandas as pd
import os
import torch.nn.functional as F


# # Reuse the model definition from training script
# class CNNRNN(nn.Module):
#     def __init__(self, n_mels=32, hidden_size=128, num_layers=2):
#         super(CNNRNN, self).__init__()
#         self.conv = nn.Sequential(
#             nn.Conv1d(n_mels, 64, kernel_size=3, padding=1),
#             nn.ReLU(),
#             nn.Conv1d(64, 32, kernel_size=3, padding=1),
#             nn.ReLU()
#         )
#         self.rnn = nn.LSTM(
#             input_size=32,
#             hidden_size=hidden_size,
#             num_layers=num_layers,
#             batch_first=True,
#             bidirectional=True
#         )
#         self.fc = nn.Sequential(
#             nn.Linear(hidden_size * 2, 64),
#             nn.ReLU(),
#             nn.Dropout(0.3),
#             nn.Linear(64, NUM_CLASSES)
#         )
# 
#     def forward(self, x, mask):
#         x = self.conv(x)
#         x = x.permute(0, 2, 1)
#         x, _ = self.rnn(x)
#         x = self.fc(x)
#         x = x * mask.unsqueeze(-1)
#         return x

def prepare_input(audio_path, interval_width=20):
    # Load audio
    waveform, sr = torchaudio.load(audio_path)

    # Ensure mono channel
    waveform = waveform.mean(dim=0, keepdim=True)

    # Compute mel spectrogram
    mel_spec_transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=sr,
        hop_length=int(sr * interval_width / 1000),
        n_mels=32
    )
    mel_spec = mel_spec_transform(waveform)
    mel_spec = mel_spec.squeeze(0)

    # Normalization (same as in training)
    mel_spec = torch.log(mel_spec + 1e-9)
    mel_spec = (mel_spec - mel_spec.mean()) / (mel_spec.std() + 1e-9)

    return mel_spec, sr

def inference(model_path, audio_path, output_csv_path):
    # Load the trained model
    model = CNNRNN()
    model.load_state_dict(torch.load(model_path, weights_only=True))
    model.eval()

    # Prepare input
    mel_spec, sr = prepare_input(audio_path)

    # Prepare batch dimensions
    mel_spec = mel_spec.unsqueeze(0)  # Add batch dimension

    # Create mask
    mask = (mel_spec != 0).any(dim=1).float()

    # Inference
    with torch.no_grad():
        outputs = model(mel_spec, mask)

    # Convert outputs to probabilities
    probabilities = torch.sigmoid(outputs.squeeze())

    # Convert to numpy for easier handling
    probs_np = probabilities.numpy()

    # Create DataFrame
    df = pd.DataFrame({
        'time_ms': [i * interval_width for i in range(len(probs_np))],
        'break_probability': probs_np[:, 1],  # Probability of being a break
        'is_break': (probs_np[:, 1] > 0.5).astype(int)  # Threshold at 0.5
    })

    # Save to CSV
    df.to_csv(output_csv_path, index=False)

    print(f"Inference completed. Results saved to {output_csv_path}")
    return df

# # Example usage
# if __name__ == "__main__":
#     # Paths
#     MODEL_PATH = 'best_model.pth'
#     AUDIO_PATH = '../../data_processing/separate_scripts/golden_audio/16.mp3'
#     OUTPUT_CSV_PATH = 'break_predictions.csv'
# 
#     # Run inference
#     results = inference(MODEL_PATH, AUDIO_PATH, OUTPUT_CSV_PATH)
# 
#     # Optional: print some statistics
#     print(f"Total predictions: {len(results)}")
#     print(f"Number of predicted breaks: {results['is_break'].sum()}")


In [8]:
# for an entire directory of audio (all following the name format of {INTEGER}.mp3), run inference on all of them and put their corresponding csvs in a directory
input_dir = '../../data/clean/audio/vocals'
output_dir = 'predictions'

def inference_dir(model_path, input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    for audio_file in os.listdir(input_dir):
        if not audio_file.endswith('.mp3'):
            continue
        audio_path = os.path.join(input_dir, audio_file)
        output_csv_path = os.path.join(output_dir, f"{audio_file[:-4]}_predictions.csv")
        inference(model_path, audio_path, output_csv_path)
        
# Run inference on the directory
inference_dir('best_model.pth', input_dir, output_dir)

Inference completed. Results saved to predictions/0_predictions.csv
Inference completed. Results saved to predictions/10_predictions.csv
Inference completed. Results saved to predictions/11_predictions.csv
Inference completed. Results saved to predictions/12_predictions.csv
Inference completed. Results saved to predictions/13_predictions.csv
Inference completed. Results saved to predictions/14_predictions.csv
Inference completed. Results saved to predictions/15_predictions.csv
Inference completed. Results saved to predictions/16_predictions.csv
Inference completed. Results saved to predictions/17_predictions.csv
Inference completed. Results saved to predictions/18_predictions.csv
Inference completed. Results saved to predictions/19_predictions.csv
Inference completed. Results saved to predictions/1_predictions.csv
Inference completed. Results saved to predictions/20_predictions.csv
Inference completed. Results saved to predictions/21_predictions.csv
Inference completed. Results saved t

In [9]:
import pandas as pd
import os

# Get all prediction files
pred_dir = 'predictions'
break_dir = 'breaks'
os.makedirs(break_dir, exist_ok=True)

for filename in os.listdir(pred_dir):
    if filename.endswith('_predictions.csv'):
        # Read predictions file
        pred_df = pd.read_csv(os.path.join(pred_dir, filename))

        # Create breaks dataframe
        breaks_df = pd.DataFrame({
            'start': pred_df['time_ms'],
            'end': pred_df['time_ms'] + 20,
            'break': pred_df['is_break']
        })

        # Get the integer prefix from filename
        prefix = filename.split('_')[0]

        # Save to breaks directory
        output_filename = f'{prefix}_breaks.csv'
        breaks_df.to_csv(os.path.join(break_dir, output_filename), index=False)


In [10]:
import pandas as pd
import os

break_dir = 'breaks'
all_breaks = []

for filename in sorted(os.listdir(break_dir)):
    if filename.endswith('_breaks.csv'):
        # Read break file
        df = pd.read_csv(os.path.join(break_dir, filename))

        # Extract file number
        file_num = int(filename.split('_')[0])

        # Add file column
        df['file'] = file_num

        all_breaks.append(df)

# Concatenate all dataframes
if all_breaks:
    final_df = pd.concat(all_breaks, ignore_index=True)

    # Reorder columns and add index
    final_df = final_df.reset_index().rename(columns={'index': 'index'})
    final_df = final_df[['index', 'file', 'start', 'end', 'break']]
    
    # remove last row if the last row itself contains any null values
    if final_df.iloc[-1].isnull().values.any():
        final_df = final_df[:-1]
    
    # Save to csv
    final_df.to_csv('segment_breaks.csv', index=False)
